In [2]:
!pip3 install hmmlearn 


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3.10 -m pip install --upgrade pip


In [3]:
import pandas as pd
from hmmlearn import hmm
import numpy as np

In [4]:
df = pd.read_csv('dataset/events.csv', sep=',')
df

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN
...,...,...,...,...,...
2756096,1438398785939,591435,view,261427,NaN
2756097,1438399813142,762376,view,115946,NaN
2756098,1438397820527,1251746,view,78144,NaN
2756099,1438398530703,1184451,view,283392,NaN


In [5]:
# Sort entries by visitorid and time
df = df.sort_values(by=['visitorid', 'timestamp'])
# Change time to readable date & time 
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
df

,timestamp,visitorid,event,itemid,transactionid
1361687,2015-09-11 20:49:49.439,0,view,285930,NaN
1367212,2015-09-11 20:52:39.591,0,view,357564,NaN
1367342,2015-09-11 20:55:17.175,0,view,67045,NaN
830385,2015-08-13 17:46:06.444,1,view,72028,NaN
742616,2015-08-07 17:51:44.567,2,view,325215,NaN
...,...,...,...,...,...
206556,2015-06-10 21:46:08.922,1407575,view,121220,NaN
47311,2015-06-03 15:01:29.991,1407576,view,356208,NaN
1762583,2015-05-17 21:48:04.867,1407577,view,427784,NaN
1744277,2015-05-17 01:21:23.288,1407578,view,188736,NaN


In [6]:
df['is_purchase'] = (df['event'] == 'transaction').astype(int)
df

,timestamp,visitorid,event,itemid,transactionid,is_purchase
1361687,2015-09-11 20:49:49.439,0,view,285930,NaN,0
1367212,2015-09-11 20:52:39.591,0,view,357564,NaN,0
1367342,2015-09-11 20:55:17.175,0,view,67045,NaN,0
830385,2015-08-13 17:46:06.444,1,view,72028,NaN,0
742616,2015-08-07 17:51:44.567,2,view,325215,NaN,0
...,...,...,...,...,...,...
206556,2015-06-10 21:46:08.922,1407575,view,121220,NaN,0
47311,2015-06-03 15:01:29.991,1407576,view,356208,NaN,0
1762583,2015-05-17 21:48:04.867,1407577,view,427784,NaN,0
1744277,2015-05-17 01:21:23.288,1407578,view,188736,NaN,0


In [7]:
sequences = df.groupby(['visitorid','itemid'])['event'].apply(list)

In [8]:
sequences

visitorid  itemid
0          67045           [view]
           285930          [view]
           357564          [view]
1          72028           [view]
2          216305    [view, view]
                         ...     
1407575    121220          [view]
1407576    356208          [view]
1407577    427784          [view]
1407578    188736          [view]
1407579    2521            [view]
Name: event, Length: 2145179, dtype: object

## Baum-Welch Algorithm

In [9]:
print(sequences[5])

itemid
61396    [view]
Name: event, dtype: object


In [ ]:
# Define states 
states = {'casual_browsing': 0, 'comparing': 1, 'ready_to_purchase': 2}
observations = {'view': 0, 'addtocart': 1, 'transaction': 2}
pi = np.array([0.7, 0.2, 0.1])

# Define traning sequence
train_seq = []
for sequence in sequences:
    sequence = [observations[obv] for obv in sequence]
    train_seq.append(sequence)


# Initialize HMM
model = hmm.CategoricalHMM(n_components=3, n_iter=100)

# Set init probs
model.startprob_ = pi

model.init_params = 'te' 

# Flatten
train_seq_fixed = np.concatenate(train_seq).reshape(-1, 1)
# Lengths
lengths = [len(seq) for seq in sequences]

model.fit(train_seq_fixed, lengths)

print("Transmission Matrix (A):\n", model.transmat_)
print("Emission Matrix (B):\n", model.emissionprob_)